### Optional: train with SageMaker built-in algorithms
Instead of developing your own script, you can use one of the SageMaker [built-in algorithms](https://docs.aws.amazon.com/sagemaker/latest/dg/algos.html). In this section you use the [XGBoost](https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost.html) algorithm. You don't need to provide your own training script. You only need to instantiate a `ModelTrainer` with the built-in algorithm image, set [hyperparameters](https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost_hyperparameters.html), and call `train()`.

<div class="alert alert-info">This is an optional step and demonstrates how to train a model using a built-in SageMaker algorithm without writing a training script. You don't need to execute this section if you're on a time budget.</div>

In [ ]:
# get training container uri
from sagemaker.core.image_uris import retrieve as retrieve_image_uri

training_image = retrieve_image_uri(framework='xgboost', version='1.7-1', region=region)

print(training_image)

In [ ]:
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import Compute
from sagemaker.core.shapes import OutputDataConfig, StoppingCondition

# Instantiate a ModelTrainer for the built-in XGBoost algorithm
builtin_trainer = ModelTrainer(
    training_image=training_image,
    compute=Compute(
        instance_type=train_instance_type,
        instance_count=train_instance_count,
    ),
    role=sm_role,
    base_job_name="from-idea-to-prod-training",
    output_data_config=OutputDataConfig(s3_output_path=output_s3_url),
    hyperparameters={
        'num_round': 50,
        'max_depth': 3,
        'eta': 0.5,
        'alpha': 2.5,
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_weight': 3,
        'early_stopping_rounds': 10,
        'verbosity': 1,
    },
    stopping_condition=StoppingCondition(max_runtime_in_seconds=20 * 60),
)


In [ ]:
builtin_trainer.hyperparameters

In [ ]:
# helper function to load XGBoost model into xgboost.Booster
def load_model(model_data_s3_uri):
    import xgboost as xgb
    import tarfile
    import pickle as pkl

    model_file = "./xgboost-model.tar.gz"
    bucket, key = model_data_s3_uri.replace("s3://", "").split("/", 1)
    boto3.client("s3").download_file(bucket, key, model_file)
    
    with tarfile.open(model_file, "r:gz") as t:
        t.extractall(path=".")
    
    # Load model
    model = xgb.Booster()
    model.load_model("xgboost-model")

    return model

Run the training:

In [ ]:
from sagemaker.core.shapes import Channel, DataSource, S3DataSource

# Built-in algorithm mode requires Channel with content_type
builtin_input_data = [
    Channel(
        channel_name="train",
        data_source=DataSource(
            s3_data_source=S3DataSource(
                s3_data_type="S3Prefix",
                s3_uri=train_s3_url,
                s3_data_distribution_type="FullyReplicated",
            )
        ),
        content_type="text/csv",
    ),
    Channel(
        channel_name="validation",
        data_source=DataSource(
            s3_data_source=S3DataSource(
                s3_data_type="S3Prefix",
                s3_uri=validation_s3_url,
                s3_data_distribution_type="FullyReplicated",
            )
        ),
        content_type="text/csv",
    ),
]

mlflow.set_experiment(experiment_name=experiment_name)
with mlflow.start_run(
    run_name=f"container-training-{strftime(chr(37)+"d-"+chr(37)+"H-"+chr(37)+"M-"+chr(37)+"S", gmtime())}",
    description="training in the notebook 02 with a training job") as run:
    mlflow.log_params(builtin_trainer.hyperparameters)
        
    builtin_trainer.train(
        input_data_config=builtin_input_data,
        wait=True,
        logs=False,
    )
    
    # Access training job via private attribute
    builtin_training_job = builtin_trainer._latest_training_job
    
    mlflow.set_tags(
        {
            'mlflow.user':user_profile_name,
            'mlflow.source.name':f'https://{region}.console.aws.amazon.com/sagemaker/home?region={region}#/jobs/{builtin_training_job.training_job_name}',
            'mlflow.source.type':'JOB'
        }
    )
    mlflow.log_param("training job name", builtin_training_job.training_job_name)
    mlflow.xgboost.log_model(load_model(builtin_training_job.model_artifacts.s3_model_artifacts), artifact_path="model")


See now the details of the training job by clicking on the link constructed by the cell below:

In [ ]:
# Show the training job link
display(
    HTML('<b>See the SageMaker <a target="top" href="https://{}.console.aws.amazon.com/sagemaker/home?region={}#/jobs/{}">training job</a></b>'.format(
            region, region, builtin_training_job.training_job_name))
)

#### Output model performance from the estimator

In [ ]:
if builtin_training_job:
    training_job_name = builtin_training_job.training_job_name

In [ ]:
%store training_job_name

In [ ]:
metrics = None
while not metrics:
    metrics = sm.describe_training_job(
        TrainingJobName=training_job_name
        ).get("FinalMetricDataList")

    if not metrics:
        print(f"Training job {training_job_name} hasn't finished yet!")
        time.sleep(10)
    
train_auc = float([m['Value'] for m in metrics if m['MetricName'] == 'train:auc'][0])
validate_auc = float([m['Value'] for m in metrics if m['MetricName'] == 'validation:auc'][0])

print(f"Train-auc:{train_auc:.4f}, Validate-auc:{validate_auc:.4f}")

In [ ]:
# Print the S3 path to the model artifact:
builtin_training_job.model_artifacts.s3_model_artifacts

#### Optional: deploy from a built-in estimator
<div class="alert alert-info">This section is optional – you don't need it for the further course of the workshop. To run this session you need to run a SageMaker built-in algorithm training job in the previous section <b>Optional: train with SageMaker built-in algorithms</b>.</div>

If you have a trained estimator object, you can deploy the model in one line of code by using [`deploy()`](https://sagemaker.readthedocs.io/en/stable/api/training/estimators.html#sagemaker.estimator.Estimator.deploy) method of the estimator. This section shows how to do this.

In [ ]:
# take the built-in ModelTrainer that you trained in the previous section
builtin_training_job.training_job_name

In [ ]:
# create a real-time endpoint from the built-in trainer
endpoint_name_from_estimator = f"from-idea-to-prod-endpoint-estimator-{strftime('%d-%H-%M-%S', gmtime())}"

builtin_model_builder = ModelBuilder(
    model=builtin_trainer,
    role_arn=sm_role,
    instance_type='ml.m5.large',
)
builtin_model = builtin_model_builder.build()
builtin_model_name = builtin_model.model_name

# Deploy with DataCaptureConfig via boto3 (workaround for SDK V3 bug)
sm_client = boto3.client('sagemaker')
sm_client.create_endpoint_config(
    EndpointConfigName=endpoint_name_from_estimator,
    ProductionVariants=[
        {
            'VariantName': 'AllTraffic',
            'ModelName': builtin_model_name,
            'InstanceType': 'ml.m5.large',
            'InitialInstanceCount': 1,
        }
    ],
    DataCaptureConfig={
        'EnableCapture': True,
        'InitialSamplingPercentage': 100,
        'DestinationS3Uri': f's3://{bucket_name}/{bucket_prefix}/data-capture',
        'CaptureOptions': [
            {'CaptureMode': 'Input'},
            {'CaptureMode': 'Output'},
        ],
    },
)
sm_client.create_endpoint(
    EndpointName=endpoint_name_from_estimator,
    EndpointConfigName=endpoint_name_from_estimator,
)

# Wait for endpoint
waiter = sm_client.get_waiter('endpoint_in_service')
waiter.wait(EndpointName=endpoint_name_from_estimator)

In [ ]:
# load test data as CSV files
test_x = pd.read_csv("tmp/test_x.csv", header=None)
test_y = pd.read_csv("tmp/test_y.csv", names=['y'])

In [ ]:
# Predict using the real-time endpoint via boto3
sm_runtime = boto3.client('sagemaker-runtime')

csv_data = '\n'.join([','.join(map(str, row)) for row in test_x.values])

response = sm_runtime.invoke_endpoint(
    EndpointName=endpoint_name_from_estimator,
    ContentType='text/csv',
    Accept='text/csv',
    Body=csv_data,
)
predictions = np.array(response['Body'].read().decode('utf-8').strip().split('\n'), dtype=float).squeeze()
predictions

In [ ]:
test_results = pd.concat(
    [
        pd.Series(predictions, name="y_pred", index=test_x.index),
        test_x,
    ],
    axis=1,
)
test_results.head()

In [ ]:
pd.crosstab(
    index=test_y['y'].values,
    columns=np.round(predictions), 
    rownames=['actuals'], 
    colnames=['predictions']
)

In [ ]:
test_auc = roc_auc_score(test_y, test_results["y_pred"])
print(f"Test-auc: {test_auc:.4f}")